Joyce Wang

02/17/2026

# Gene Set Enrichment Analysis with GSEApy: Head-to-Head Comparisons

In [1]:
# Core libraries
import hisepy
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns

from gseapy import dotplot

/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/VertexPartition.py:413: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\frac{1}{m} \\sum_{ij} \\left(A_{ij} - \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/VertexPartition.py:788: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\sum_{ij} \\left(A_{ij} - \\gamma \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/Optimiser.py:27: SyntaxWarning: invalid escape sequence '\g'
  implementation therefore does not guarantee subpartition :math:`\gamma`-density.
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/Optimiser.py:346: SyntaxWarning: invalid escape sequence '\s'
  .. math:: Q = \sum_k \\lambda_k Q_k.


# Load in DEG Results for head-to-head comparisons between formulations for each cell type

In [2]:
file_id = ['5a715df8-b774-4947-a023-5fd9c9d97e19']

# dict
file_path = hisepy.read_files(file_list=file_id)

In [3]:
# dict values --> list
deg_dfs = list(file_path.values())

In [4]:
# concatenate
deg_df = pd.concat(deg_dfs, ignore_index=True)

deg_df

,projectGuid,mergeKey,emr,labLastModified,lastUpdated,surveyLastModified,surveyScheme,cohort.cohortGuid,file.id,file.name,...,Unnamed: 0,names,scores,logfoldchanges,pvals,pvals_adj,cell_type,form_1,form_2,filename
0,910ae58c-41c3-49fb-8e48-08fb25529a61,5a715df8-b774-4947-a023-5fd9c9d97e19_00000000-...,,0001-01-01T00:00:00Z,2026-03-03T19:20:33.242Z,0001-01-01T00:00:00Z,,,5a715df8-b774-4947-a023-5fd9c9d97e19,/home/workspace/input/3733913762/2025_bgmp/5a7...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,CD44,9.302001,0.051674,1.378264e-20,1.842739e-17,CD4 Central Memory,Afatinib,Afatinib dimaleate,/home/workspace/input/3733913762/2025_bgmp/5a7...
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,CD2,8.935438,0.094797,4.055687e-19,2.711227e-16,CD4 Central Memory,Afatinib,Afatinib dimaleate,/home/workspace/input/3733913762/2025_bgmp/5a7...
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,S100A10,7.404135,0.028103,1.320082e-13,5.673898e-11,CD4 Central Memory,Afatinib,Afatinib dimaleate,/home/workspace/input/3733913762/2025_bgmp/5a7...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83437,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1324.0,SELL,-2.734802,-0.350373,6.241792e-03,6.380187e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...
83438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1325.0,AKT3,-2.773036,-0.285096,5.553597e-03,6.150609e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...
83439,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1326.0,PRKCQ,-2.781397,-0.195473,5.412555e-03,6.150609e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...
83440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1327.0,RIPOR2,-2.910409,-0.245898,3.609560e-03,6.150609e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...


## Clean up the DEG dataframe

In [5]:
# Remove empty rows
deg_df = deg_df.dropna(subset=["names"])

In [6]:
# Keep only columns needed for GSEA
deg_df = deg_df[[
    "cell_type",
    "form_1",
    "form_2",
    "names",
    "logfoldchanges",
    "pvals",
    "pvals_adj"
]]

In [7]:
# Adding a comparison label column
deg_df["comparison"] = deg_df["form_1"] + "_vs._" + deg_df["form_2"]

# Gene set `MSigDB_Hallmark_2020` for Gene Set Enrichment Analysis
- `MSigDB_Hallmark_2020` has fewer pathways.

In [8]:
from gseapy import Msigdb

In [9]:
### Add a dict to store results
gsea_res = {}

for cell_type, cell_type_df in deg_df.groupby("cell_type"):
    ### Nested dict to store each drug
    gsea_res[cell_type] = {}

    for comparison, comparison_df in cell_type_df.groupby("comparison"):
        
        # Build a ranked gene list: extract ranked genes (names + logFC)
        gene_rank = comparison_df[["names", "logfoldchanges"]]

        # With GSEA of drug treatments vs. DMSO control, previously filtered genes expressed in at least 30 cells with `sc.pp.calculate_qc_metrics` and `subset_cell.var.n_cells_by_counts()`.
        
        # Build a ranked gene list: Sort the genes from high to low fold changes
        gene_rank = gene_rank.sort_values("logfoldchanges", ascending=False)
        
        # Run prerank GSEA with gp.prerank()
        pre_res = gp.prerank(
            rnk=gene_rank,
            gene_sets = "MSigDB_Hallmark_2020",
            # default min_size = 15 -- note, number of features is reduced already by limited gene panel
            outdir = None, # don't write to disk
            verbose = False # make True to see what's going on behind the scenes
        )

        ### Store the output in the dict
        gsea_res[cell_type][comparison] = pre_res

## View results for each cell type and comparison, in a table

In [10]:
# View dictionary key to see what comparisons were made
gsea_res["CD4 Naive"].keys()

dict_keys(['Afatinib_vs._Afatinib dimaleate', 'Baricitinib_vs._Baricitinib phosphate', 'Canertinib_vs._Canertinib dihydrochloride', 'Erlotinib_vs._Erlotinib hydrochloride', 'Gefitinib_vs._Gefitinib hydrochloride', 'NVP-BSK805_vs._NVP-BSK805 2HCl', 'Ruxolitinib_vs._Ruxolitinib phosphate', 'Tofacitinib citrate_vs._Tofacitinib'])

In [11]:
# View results for each cell type and comparison (8 total comparisons!)
pre_res = gsea_res['CD4 Naive']['Tofacitinib citrate_vs._Tofacitinib']
pre_res.res2d.sort_values('FDR q-val').head(10)

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
6,prerank,E2F Targets,0.37378,1.311613,0.085072,0.443614,0.876,23/66,22.68%,MYC;PLK1;MTHFD2;CDK1;NOLC1;JPT1;CENPE;TP53;TFR...
16,prerank,Mitotic Spindle,0.38823,1.134437,0.304196,0.47226,0.998,9/26,21.57%,PLK1;CDK1;CENPE;NCK2;SMC1A;NDC80;RASA1;SMC3;HDAC6
15,prerank,Inflammatory Response,0.340196,1.145345,0.256956,0.483283,0.997,15/52,15.89%,IL2RB;MYC;IRF1;ICAM1;PVR;HIF1A;IL18R1;TLR2;ADG...
12,prerank,UV Response Up,0.383954,1.177022,0.232014,0.493876,0.989,14/35,27.48%,IRF1;PARP2;ICAM1;RAB27A;JUNB;TFRC;FKBP4;EIF5;B...
10,prerank,Myc Targets V1,0.42342,1.21659,0.196013,0.499081,0.971,11/24,33.71%,MYC;NOLC1;C1QBP;PWP1;MCM2;TOMM70;USP1;CDK2;HDA...
9,prerank,mTORC1 Signaling,0.366282,1.237833,0.141623,0.502372,0.956,24/53,34.03%,DDIT4;PLK1;MTHFD2;NFIL3;TFRC;GLRX;MLLT11;NAMPT...
7,prerank,Adipogenesis,0.439258,1.261199,0.165552,0.507261,0.939,10/23,31.39%,BCL6;HADH;REEP5;VEGFB;IFNGR1;TANK;SCP2;ETFB;PG...
14,prerank,Androgen Response,0.433813,1.145513,0.28381,0.523364,0.997,3/17,7.51%,ARID5B;PTK2B;RRP12
5,prerank,G2-M Checkpoint,0.394666,1.313536,0.096346,0.526125,0.871,11/48,14.38%,MYC;PLK1;CDK1;NOLC1;JPT1;HIF1A;CENPE;ATF5;SMC1...
11,prerank,Apical Junction,0.379356,1.1828,0.22335,0.529806,0.987,12/36,26.04%,ICAM1;CD274;HADH;MAPK11;ITGA2;RAC2;GNAI1;RASA1...


# Create data frame, data frame to CSV, upload to HISE

In [12]:
# empty list
all_results = []

for cell_type in gsea_res:
    for comparison in gsea_res[cell_type]:
        res_df = gsea_res[cell_type][comparison].res2d  # results in rows with columns like Term, NES, FDR q-val, etc...

        # Add more columns for detail
        res_df["cell_type"] = cell_type
        res_df["comparison"] = comparison
        res_df["gene_set"] = "Hallmark"   # Hallmark, KEGG, or Reactome

        # Add to dataframe
        all_results.append(res_df)

# Create a dataframe
final_gsea_df = pd.concat(all_results)

In [13]:
final_gsea_df

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes,cell_type,comparison,gene_set
0,prerank,Spermatogenesis,-0.671652,-1.681717,0.003567,0.061646,0.058,9/15,19.67%,MAP7;MAST2;PARP2;IDE;BRAF;PSMG1;TOPBP1;MTOR;CDK1,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,Hallmark
1,prerank,Epithelial Mesenchymal Transition,-0.531484,-1.514846,0.020563,0.2435,0.371,8/31,9.50%,VEGFA;COL1A1;TPM1;GPC1;ITGA2;TGFBI;PLAUR;MEST,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,Hallmark
2,prerank,Glycolysis,-0.495287,-1.476768,0.022317,0.24761,0.499,14/40,19.67%,VEGFA;SLC25A13;SLC37A4;GPC1;TGFBI;GALE;PGAM1;E...,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,Hallmark
3,prerank,Interferon Alpha Response,0.342196,1.431281,0.012987,0.069571,0.1,7/36,7.11%,LAMP3;OAS1;IFITM1;MX1;IRF2;SELL;IFIH1,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,Hallmark
4,prerank,Pperoxisome,-0.532686,-1.411408,0.052692,0.349582,0.71,13/20,29.69%,TOP2A;ITGB1BP1;IDE;HRAS;MSH2;SMARCC1;ACAA1;ABC...,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,Hallmark
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32,prerank,Myc Targets V1,-0.253032,-0.713647,0.863481,1.0,1.0,5/24,20.16%,MCM2;RNPS1;TFDP1;TOMM70;HPRT1,Treg,Tofacitinib citrate_vs._Tofacitinib,Hallmark
33,prerank,heme Metabolism,-0.255258,-0.711167,0.880911,0.969247,1.0,2/22,3.06%,ABCB6;XPO7,Treg,Tofacitinib citrate_vs._Tofacitinib,Hallmark
34,prerank,Androgen Response,0.265108,0.692678,0.858824,0.915117,1.0,4/17,17.02%,INPP4B;TMEM50A;ARID5B;INSIG1,Treg,Tofacitinib citrate_vs._Tofacitinib,Hallmark
35,prerank,Oxidative Phosphorylation,-0.23618,-0.658053,0.921502,0.977608,1.0,12/24,32.71%,SLC25A4;ETFB;HSD17B10;TOMM70;PDHX;ATP1B1;OXA1L...,Treg,Tofacitinib citrate_vs._Tofacitinib,Hallmark


In [14]:
final_gsea_df.to_csv("il6_jak-stat_head_to_head_gseapy_hallmark.csv")

In [15]:
uuid_list = [
    '02765813-6130-4fac-8708-f01768aa05b6',
    '06b73fab-62d2-4fc3-8a86-2df960cbc1ea',
    '1aecccab-f62a-4c49-b2fe-ba5777930262',
    '207c5f6c-92ec-4690-a7fd-07b0cbebd9f6',
    '3adc31cf-1ea9-4a7d-bc63-31358cb8a325',
    '45e4bd43-4fae-49df-8974-8d043e395f73',
    '4c13d814-1493-48f8-8021-a819b556e97b',
    '56bc5070-2968-4c6b-8198-4e997c75e4fe',
    '64e7735c-e89b-41ef-a293-a39b9ed5ade9',
    '72755c82-880d-4814-8322-6a81ed07466d',
    '9693f71c-dcb3-4d40-b2ad-74fc2ad147de',
    '9f37360e-0191-418b-ab51-fdb86ab9be09',
    'a3bc4704-5fe3-4a74-bce5-0700689db1f6',
    'af42c180-0218-4918-ae12-0a4f43c4aa1f',
    'b790716c-b2de-4028-ad60-1e5a7b81f05d',
    'bfa4c2f1-69d5-4bcf-a1a6-9beb69dbffc9',
    'd076758f-3d76-49b7-91be-0217eefcdf26',
    'd2f8da49-85eb-4dea-a166-5308bb73de91'
]

file_list = hisepy.cache_files(uuid_list)

In [16]:
# a function to make unique destination strings using the periodic table elements
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
        
    rand_str = '-'.join(rand_el)
    return rand_str

In [17]:
# getting unique title for upload, using UTC time
from datetime import datetime, timezone

# getting the current time and saving it
utc_current = datetime.now(timezone.utc)
print(utc_current)

# listing the files to upload
files_to_upload = [
    "il6_jak-stat_head_to_head_gseapy_hallmark.csv"
]

# uploading the files
hisepy.upload_files(
    files = files_to_upload,
    study_space_id = "c8a94b84-b0b7-40a9-980b-81a63ad6e115",
    title = f"GSEApy_head-to-head_Hallmark_data_h5ad_{utc_current}",
    input_file_ids = uuid_list,
    #destination is randomly generated elements
    destination = element_id()
)

2026-03-04 08:12:47.720973+00:00
checking if conda environment can compile...
creating temp conda environment...
temp conda environment created successfully, now packing...
Cannot determine the current notebook.
1) /home/workspace/gseapy_head_to_head_Hallmark_JW.ipynb
2) /home/workspace/gseapy_head_to_head_KEGG_JW.ipynb
3) /home/workspace/gseapy_head_to_head_Reactome_JW.ipynb
Please select (1-3) 


 1


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '0222a587-d308-4364-9a11-22bf4e015dbd',
 'ProcessId': 'afc6f599-d2c1-4ba9-96cd-9be925c497f3',
 'WorkflowId': '3d577fa4-9522-4277-aaed-cf7aa7acffd6',
 'FileIds': ['abeab9f8-79fe-4694-87e4-cfa5bc87bc53']}